## Signal comparison on cached 1000 solutions

Positive = annotated **first** error; all other steps, including post-error steps of unknown correctness, are treated as negative in the all-steps analysis. The verified-only sensitivity analysis excludes post-error steps. `eligible` excludes unscorable boundary steps; `common` compares every method on the same steps (neither first nor last); `all_steps_neutral` assigns zero to unavailable boundary scores, showing AUC over **all 7012 steps**. Zero imputation is a convention, not evidence that boundary steps are safe. Future-dependent signals are retrospective. AUCs are descriptive, in-sample, not held-out performance.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve().parent
assert ROOT.name == "mr-gsm8k", "Run this notebook from signal_experiments/"
sys.path.insert(0, str(ROOT))
from plots import plot_first_error_roc
from score_dc_pmi import SIGNAL_TITLES, evaluate_signal_scores, load_signal_cache

steps = load_signal_cache(
    ROOT / "cache/llama_3_mr_gsm8k_original_n1000_seed42_token_logprobs.parquet",
    ROOT / "signal_experiments/cache/signals_n1000_seed42.parquet",
)
assert steps.uuid.nunique() == 1000
results = evaluate_signal_scores(steps, SIGNAL_TITLES)
display(results.pivot(index="signal", columns="subset", values="auc").round(3))
display(results[results.subset.eq("eligible")][["signal", "steps", "first_errors", "coverage"]])

common = steps.dropna(subset=list(SIGNAL_TITLES))
fig, ax, roc_rows = plot_first_error_roc(
    common.first_error, {title: common[column] for column, title in SIGNAL_TITLES.items()},
    title="Same eligible steps for every signal",
)
ax.legend(fontsize=8)
plt.show()

verified = evaluate_signal_scores(steps, SIGNAL_TITLES, verified_only=True)
display(verified.pivot(index="signal", columns="subset", values="auc").round(3))

### Initial observations (1000 cached solutions, 7012 steps)

On the common 5012 steps, raw negative generic DC-PMI reaches AUC **0.726**, generic drop **0.671**, next-step generic increase **0.655**, next-step history increase **0.653**, generic/history z-scores **0.648/0.641**, and generic/history peak prominence **0.578/0.564**. The original notebook's AUC of **0.696** for raw generic DC-PMI is based on **verified steps only**; the verified-only comparison below reproduces it. These are exploratory in-sample numbers, not a trained or validated detector. Of 508 first errors, 102 occur on the first step (unscorable by immediate difference); **none occur on the last step** in this cache. The all-steps-neutral metric assigns zero to unavailable scores; the common-subset comparison avoids that arbitrary choice. Post-error steps of unknown correctness are treated as negatives in the all-steps comparison.